In [2]:
import pandas as pd
import numpy as np
from scipy import stats

print(" Libraries geladen!")

 Libraries geladen!


## 2. Daten laden

In [4]:
# Lade manuelle Evaluation
df = pd.read_csv('human_eval_rag.csv')

## 3. Datenbereinigung & Typ-Konvertierung


In [6]:
# Konvertiere zu numerischen Werten
numeric_cols = ['em', 'f1', 'rougeL', 'bleu', 'human_correctness_0_2', 'human_faithfulness_0_2']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Entferne Zeilen mit fehlenden Werten
df_clean = df.dropna(subset=numeric_cols)

Spalten zu numerischen Werten konvertiert


## 4. Korrelationsanalyse: Correctness-Metriken

In [8]:
print("="*80)
print(" KORRELATION: AUTOMATISCHE METRIKEN vs. MANUELLE CORRECTNESS")
print("="*80)
print()

metrics = ['em', 'f1', 'rougeL', 'bleu']
results = []

for metric in metrics:
    # Pearson Korrelation
    r, p = stats.pearsonr(df_clean[metric], df_clean['human_correctness_0_2'])
    
    # Signifikanz
    if p < 0.001:
        sig = "***"
    elif p < 0.01:
        sig = "**"
    elif p < 0.05:
        sig = "*"
    else:
        sig = "n.s."
    
    # Interpretation
    if abs(r) > 0.9:
        interp = "Sehr stark"
        
    elif abs(r) > 0.7:
        interp = "Stark"
       
    elif abs(r) > 0.5:
        interp = "Moderat"
       
    elif abs(r) > 0.3:
        interp = "Schwach"
     
    else:
        interp = "Keine"
     
    
    print(f" {metric.upper():10s}  r = {r:6.3f}  p = {p:.6f} {sig:4s}  [{interp}]")
    
    results.append({
        'Metric': metric.upper(),
        'Pearson r': round(r, 3),
        'p-value': f"{p:.6f}" if p >= 0.001 else "<0.001",
        'Signifikanz': sig,
        'Interpretation': interp
    })

print()
print("Legende: *** p<0.001, ** p<0.01, * p<0.05, n.s. = nicht signifikant")
print("="*80)

 KORRELATION: AUTOMATISCHE METRIKEN vs. MANUELLE CORRECTNESS

 EM          r =  0.851  p = 0.000000 ***   [Stark]
 F1          r =  0.948  p = 0.000000 ***   [Sehr stark]
 ROUGEL      r =  0.948  p = 0.000000 ***   [Sehr stark]
 BLEU        r =  0.252  p = 0.002079 **    [Keine]

Legende: *** p<0.001, ** p<0.01, * p<0.05, n.s. = nicht signifikant


## 5. Ergebnis-Tabelle

In [10]:
# Als DataFrame
results_df = pd.DataFrame(results)

print(" ERGEBNISTABELLE (für Table 4.7 in Thesis):")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

 ERGEBNISTABELLE (für Table 4.7 in Thesis):
Metric  Pearson r  p-value Signifikanz Interpretation
    EM      0.851   <0.001         ***          Stark
    F1      0.948   <0.001         ***     Sehr stark
ROUGEL      0.948   <0.001         ***     Sehr stark
  BLEU      0.252 0.002079          **          Keine


## 6. Korrelation: Human Correctness vs. Human Faithfulness

In [12]:
r_human, p_human = stats.pearsonr(df_clean['human_correctness_0_2'], 
                                   df_clean['human_faithfulness_0_2'])

print("\n" + "="*80)
print(" KORRELATION: HUMAN CORRECTNESS vs. HUMAN FAITHFULNESS")
print("="*80)
print(f" r = {r_human:.3f}, p = {p_human:.6f}")
print(" Interpretation: Moderate Korrelation")
print("   → Correctness und Faithfulness sind verwandt, aber nicht identisch")
print("   → Bestätigt Notwendigkeit multi-dimensionaler Evaluation")
print("="*80)


 KORRELATION: HUMAN CORRECTNESS vs. HUMAN FAITHFULNESS
 r = 0.558, p = 0.000000
 Interpretation: Moderate Korrelation
   → Correctness und Faithfulness sind verwandt, aber nicht identisch
   → Bestätigt Notwendigkeit multi-dimensionaler Evaluation


## 7. Deskriptive Statistiken

In [14]:
print("\n DESKRIPTIVE STATISTIKEN")
print("\n Automatische Metriken:")
print(df_clean[['em', 'f1', 'rougeL', 'bleu']].describe().round(3))


 DESKRIPTIVE STATISTIKEN

 Automatische Metriken:
            em       f1   rougeL     bleu
count  147.000  147.000  147.000  147.000
mean     0.136    0.172    0.172    0.014
std      0.344    0.357    0.357    0.116
min      0.000    0.000    0.000    0.000
25%      0.000    0.000    0.000    0.000
50%      0.000    0.000    0.000    0.000
75%      0.000    0.000    0.000    0.000
max      1.000    1.000    1.000    1.000


In [15]:
print(" Manuelle Bewertungen (Skala 0-2):")
print(df_clean[['human_correctness_0_2', 'human_faithfulness_0_2']].describe().round(3))

 Manuelle Bewertungen (Skala 0-2):
       human_correctness_0_2  human_faithfulness_0_2
count                147.000                 147.000
mean                   0.374                   0.830
std                    0.760                   0.975
min                    0.000                   0.000
25%                    0.000                   0.000
50%                    0.000                   0.000
75%                    0.000                   2.000
max                    2.000                   2.000


## 8. Export für Thesis

In [17]:
# Export als CSV
results_df.to_csv('correlation_results.csv', index=False)
print(" CSV gespeichert: correlation_results.csv")



 CSV gespeichert: correlation_results.csv
